In [0]:
# =============================================================================
# 01 - DEPENDENCIAS
# =============================================================================

%pip install --upgrade \
    requests==2.32.3 \
    beautifulsoup4==4.12.3 \
    lxml==5.3.0 \
    tenacity==9.0.0

# Reinicia el intérprete para garantizar que las versiones instaladas
# sean las utilizadas por las celdas posteriores.
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# =============================================================================
# 02 - IMPORTACIONES Y LOGGING
# =============================================================================

import re
import json
import sys
import time
import random
import hashlib
import logging
import threading
import unicodedata
import importlib.metadata as importlib_metadata

from dataclasses import dataclass
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import urljoin, urlparse, unquote
from urllib.robotparser import RobotFileParser
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from bs4 import BeautifulSoup

from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log,
)

# -----------------------------------------------------------------------------
# Logging
# -----------------------------------------------------------------------------

LOG_FORMAT = "%(asctime)s | %(name)s | %(levelname)s | %(message)s"

logging.basicConfig(
    level=logging.INFO,
    format=LOG_FORMAT,
    force=True,
)

logger = logging.getLogger("bronze_ingestion")

In [0]:
# =============================================================================
# 03 - CONFIGURACIÓN GLOBAL Y FUENTES
# =============================================================================

@dataclass(frozen=True)
class SourceConfig:
    code: str
    base_url: str
    volume_name: str
    allowed_extensions: Tuple[str, ...]
    keywords: Tuple[str, ...] = ()
    exclude_keywords: Tuple[str, ...] = ()


# -----------------------------------------------------------------------------
# Versionado
# -----------------------------------------------------------------------------

NOTEBOOK_VERSION = "1.1.0"
MANIFEST_SCHEMA_VERSION = "2.0"
STORAGE_LAYOUT_VERSION = "2.0"


# -----------------------------------------------------------------------------
# Unity Catalog
# -----------------------------------------------------------------------------

CATALOG = "camaronera_2026"
BRONZE_SCHEMA = "bronce"


# -----------------------------------------------------------------------------
# Ventana temporal de adquisición
# -----------------------------------------------------------------------------

CURRENT_YEAR = datetime.now(timezone.utc).year

TARGET_YEARS = tuple(
    range(
        CURRENT_YEAR,
        CURRENT_YEAR - 3,
        -1
    )
)


# -----------------------------------------------------------------------------
# Configuración HTTP
# -----------------------------------------------------------------------------

MAX_WORKERS = 4

REQUEST_TIMEOUT = (10, 120)

REQUEST_DELAY_SECONDS = (0.8, 1.8)

USER_AGENT = "AcademicResearch-ShrimpBI/1.1"


# -----------------------------------------------------------------------------
# Política robots.txt
# -----------------------------------------------------------------------------

RESPECT_ROBOTS_TXT = True
ROBOTS_FAIL_OPEN = True


# -----------------------------------------------------------------------------
# Fuentes
# -----------------------------------------------------------------------------

SOURCES: Dict[str, SourceConfig] = {

    "CNA": SourceConfig(
        code="CNA",
        base_url="https://www.cna-ecuador.com/estadisticas/",
        volume_name="datoscna",
        allowed_extensions=(
            ".xlsx",
            ".xls",
            ".csv",
            ".pdf",
        ),

        # Los archivos estadísticos observados de CNA
        # utilizan esta denominación.
        keywords=(
            "estadistica",
        ),

        # Recursos publicados en la misma página
        # pero que no forman parte del dataset analítico.
        exclude_keywords=(
            "calendario",
            "aguaje",
        ),
    ),

    "CFN": SourceConfig(
        code="CFN",
        base_url="https://www.cfn.fin.ec/bibliotecainfo/",
        volume_name="datoscfn",
        allowed_extensions=(
            ".pdf",
        ),
        keywords=(
            "camaron",
            "acuicultura",
            "pesca",
            "langostino",
            "crustaceo",
            "marisco",
            "pesquero",
        ),
    ),
}


logger.info(
    "Años objetivo: %s",
    TARGET_YEARS
)

In [0]:
# =============================================================================
# 04 - FUNCIONES AUXILIARES
# =============================================================================

def utc_now_iso() -> str:
    """Devuelve el timestamp actual en UTC en formato ISO-8601."""
    return datetime.now(timezone.utc).isoformat()


def generate_batch_id() -> str:
    """Genera un identificador único para cada ejecución."""
    return datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S%fZ"
    )


def strip_accents(value: str) -> str:
    """Elimina marcas diacríticas para comparaciones textuales."""
    normalized = unicodedata.normalize(
        "NFKD",
        value or ""
    )

    return "".join(
        ch
        for ch in normalized
        if not unicodedata.combining(ch)
    )


def normalize_text(value: str) -> str:
    """Normaliza texto para búsquedas y fingerprints."""
    value = strip_accents(value).lower()
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def safe_filename(filename: str) -> str:
    """Genera un nombre de archivo seguro sin modificar su contenido."""

    decoded = unquote(filename or "")

    name = Path(decoded).name

    name = unicodedata.normalize(
        "NFKC",
        name
    )

    name = re.sub(
        r'[<>:"/\\|?*\x00-\x1f]',
        "_",
        name
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    ).strip(" .")

    return name or "archivo_descargado"


def extract_years(value: str) -> List[int]:
    """Extrae años con formato 20XX desde una cadena."""
    return [
        int(year)
        for year in re.findall(
            r"\b(20\d{2})\b",
            value or ""
        )
    ]


def infer_document_year(
    url: str,
    anchor_text: str = ""
) -> Optional[int]:
    """
    Determina el año documental priorizando:
    1. nombre de archivo;
    2. URL;
    3. texto visible del enlace.
    """

    filename = unquote(
        Path(
            urlparse(url).path
        ).name
    )

    years_in_filename = extract_years(filename)

    if years_in_filename:
        return years_in_filename[0]

    years_in_url = extract_years(
        unquote(url)
    )

    if years_in_url:
        return years_in_url[0]

    years_in_text = extract_years(
        anchor_text
    )

    if years_in_text:
        return years_in_text[0]

    return None


def compute_sha256_and_size(
    file_path: Path
) -> Tuple[str, int]:
    """
    Calcula SHA-256 y tamaño del archivo
    mediante lectura por bloques.
    """

    sha256 = hashlib.sha256()
    size = 0

    with open(file_path, "rb") as file_handle:

        for chunk in iter(
            lambda: file_handle.read(1024 * 1024),
            b""
        ):
            sha256.update(chunk)
            size += len(chunk)

    return sha256.hexdigest(), size


def sha256_text(value: str) -> str:
    """Calcula SHA-256 sobre una cadena UTF-8."""

    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()

def resource_id_from_url(
    url: str,
    length: int = 12
) -> str:
    """
    Genera un identificador determinista de recurso
    a partir de su URL de procedencia.

    Permite distinguir documentos con el mismo
    nombre de archivo publicados bajo URLs distintas.
    """

    return sha256_text(url)[:length]

def build_structural_fingerprint(
    source_url: str,
    visible_text: str,
    discovered_urls: List[str],
) -> Dict[str, Any]:
    """
    Construye una firma estructural reproducible de la página fuente.
    """

    normalized_visible_text = normalize_text(
        visible_text
    )

    stable_urls = sorted(
        set(discovered_urls)
    )

    return {

        "source_url":
            source_url,

        "visible_text_sha256":
            sha256_text(
                normalized_visible_text
            ),

        "url_set_sha256":
            sha256_text(
                "\n".join(stable_urls)
            ),

        "total_discovered_urls":
            len(stable_urls),

        "discovered_urls":
            stable_urls,

        "captured_at_utc":
            utc_now_iso(),
    }


def atomic_json_dump(
    payload: Dict[str, Any],
    destination: Path
) -> None:
    """
    Escribe JSON mediante archivo temporal y reemplazo final.
    """

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp_path = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    with open(
        temp_path,
        "w",
        encoding="utf-8"
    ) as file_handle:

        json.dump(
            payload,
            file_handle,
            indent=2,
            ensure_ascii=False
        )

    temp_path.replace(
        destination
    )

In [0]:
# =============================================================================
# 05 - UNITY CATALOG Y ALMACENAMIENTO BRONZE
# =============================================================================

def quote_identifier(
    identifier: str
) -> str:
    """Escapa identificadores utilizados en SQL."""

    return (
        "`"
        + identifier.replace("`", "``")
        + "`"
    )


def ensure_bronze_storage(
    config: SourceConfig
) -> Dict[str, Path]:
    """
    Garantiza la existencia del esquema y Volume
    correspondiente a una fuente.
    """

    catalog = quote_identifier(
        CATALOG
    )

    schema = quote_identifier(
        BRONZE_SCHEMA
    )

    volume = quote_identifier(
        config.volume_name
    )


    # Crear esquema si no existe
    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS
        {catalog}.{schema}
        """
    )


    # Crear Volume si no existe
    spark.sql(
        f"""
        CREATE VOLUME IF NOT EXISTS
        {catalog}.{schema}.{volume}
        """
    )


    # Ruta raíz del Volume
    root = Path(
        f"/Volumes/"
        f"{CATALOG}/"
        f"{BRONZE_SCHEMA}/"
        f"{config.volume_name}"
    )


    # Separar físicamente datos y metadatos
    raw_dir = (
        root
        / "raw"
    )

    metadata_dir = (
        root
        / "_metadata"
    )


    raw_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    metadata_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    if not root.exists():

        raise RuntimeError(
            f"No se pudo acceder al Volume esperado: {root}"
        )


    logger.info(
        "Almacenamiento Bronze disponible para %s: %s",
        config.code,
        root
    )


    return {

        "root":
            root,

        "raw":
            raw_dir,

        "metadata":
            metadata_dir,
    }


BRONZE_PATHS: Dict[
    str,
    Dict[str, Path]
] = {

    code:
        ensure_bronze_storage(
            config
        )

    for code, config
    in SOURCES.items()
}

In [0]:
# =============================================================================
# 06 - CLIENTE HTTP, REINTENTOS Y ROBOTS.TXT
# =============================================================================

_thread_local = threading.local()


def get_http_session() -> requests.Session:
    """
    Devuelve una sesión HTTP independiente por hilo.
    """

    if not hasattr(
        _thread_local,
        "session"
    ):

        session = requests.Session()

        session.headers.update({

            "User-Agent":
                USER_AGENT,

            "Accept":
                "*/*",

            "Connection":
                "keep-alive",
        })

        _thread_local.session = session


    return _thread_local.session


def robots_allows(
    config: SourceConfig
) -> bool:
    """
    Consulta robots.txt cuando la política está habilitada.

    Esta validación no sustituye una revisión de
    términos de uso o condiciones legales del sitio.
    """

    if not RESPECT_ROBOTS_TXT:
        return True


    robots_url = urljoin(
        config.base_url,
        "/robots.txt"
    )


    try:

        response = requests.get(
            robots_url,
            headers={
                "User-Agent":
                    USER_AGENT
            },
            timeout=(5, 20),
        )


        if response.status_code == 404:

            logger.info(
                "%s no publica robots.txt; se continúa.",
                config.code
            )

            return True


        response.raise_for_status()


        parser = RobotFileParser()

        parser.set_url(
            robots_url
        )

        parser.parse(
            response.text.splitlines()
        )


        allowed = parser.can_fetch(
            USER_AGENT,
            config.base_url
        )


        if not allowed:

            logger.error(
                "robots.txt no permite consultar %s.",
                config.base_url
            )


        return allowed


    except requests.RequestException as exc:

        if ROBOTS_FAIL_OPEN:

            logger.warning(
                "No se pudo verificar robots.txt para %s (%s). "
                "Se continúa por configuración.",
                config.code,
                exc,
            )

            return True


        logger.error(
            "No se pudo verificar robots.txt para %s: %s",
            config.code,
            exc,
        )

        return False


@retry(
    stop=stop_after_attempt(4),

    wait=wait_exponential(
        multiplier=1,
        min=2,
        max=30
    ),

    retry=retry_if_exception_type(
        requests.RequestException
    ),

    before_sleep=before_sleep_log(
        logger,
        logging.WARNING
    ),

    reraise=True,
)
def fetch_html(
    config: SourceConfig
) -> requests.Response:
    """
    Recupera la página HTML de una fuente
    aplicando timeout y reintentos.
    """

    if not robots_allows(config):

        raise PermissionError(
            f"robots.txt impide consultar "
            f"{config.base_url}"
        )


    time.sleep(
        random.uniform(
            *REQUEST_DELAY_SECONDS
        )
    )


    session = get_http_session()


    response = session.get(
        config.base_url,
        timeout=REQUEST_TIMEOUT
    )


    response.raise_for_status()


    return response

In [0]:
# =============================================================================
# 07 - DESCUBRIMIENTO DE DOCUMENTOS CNA
# =============================================================================

def discover_cna_files(
    target_years: Tuple[int, ...]
) -> Tuple[
    List[Dict[str, Any]],
    Dict[str, Any]
]:
    """
    Descubre documentos estadísticos publicados por
    la Cámara Nacional de Acuacultura (CNA).

    La función:
    - recupera una única vez la página de estadísticas;
    - identifica enlaces a archivos admitidos;
    - aplica criterios de inclusión y exclusión;
    - determina el año documental;
    - elimina URLs duplicadas;
    - construye una firma estructural de la fuente.
    """

    config = SOURCES["CNA"]

    response = fetch_html(config)

    soup = BeautifulSoup(
        response.text,
        "lxml"
    )

    normalized_keywords = tuple(
        normalize_text(keyword)
        for keyword in config.keywords
    )

    normalized_exclusions = tuple(
        normalize_text(keyword)
        for keyword in config.exclude_keywords
    )

    candidates: List[Dict[str, Any]] = []

    seen_urls = set()


    for anchor in soup.find_all(
        "a",
        href=True
    ):

        href = anchor.get("href")

        if not href:
            continue


        full_url = urljoin(
            config.base_url,
            href
        )


        parsed_path = unquote(
            urlparse(full_url).path
        )


        extension = Path(
            parsed_path
        ).suffix.lower()


        # -------------------------------------------------------------
        # Validar formato
        # -------------------------------------------------------------

        if extension not in config.allowed_extensions:
            continue


        anchor_text = anchor.get_text(
            " ",
            strip=True
        )


        combined_text = normalize_text(
            f"{full_url} {anchor_text}"
        )


        # -------------------------------------------------------------
        # Criterios de exclusión
        # -------------------------------------------------------------

        if any(
            excluded in combined_text
            for excluded in normalized_exclusions
        ):
            continue


        # -------------------------------------------------------------
        # Criterios de inclusión temática
        # -------------------------------------------------------------

        if (
            normalized_keywords
            and
            not any(
                keyword in combined_text
                for keyword in normalized_keywords
            )
        ):
            continue


        # -------------------------------------------------------------
        # Año documental
        # -------------------------------------------------------------

        document_year = infer_document_year(
            full_url,
            anchor_text
        )


        if document_year not in target_years:
            continue


        # -------------------------------------------------------------
        # URL única
        # -------------------------------------------------------------

        if full_url in seen_urls:
            continue


        seen_urls.add(full_url)


        candidates.append({

            "source":
                config.code,

            "source_page_url":
                config.base_url,

            "source_url":
                full_url,

            "resource_id":
                resource_id_from_url(
                    full_url
                ),

            "anchor_text":
                anchor_text,

            "original_filename":
                Path(
                    parsed_path
                ).name,

            "document_year":
                document_year,

            "extension":
                extension,
        })


    fingerprint = build_structural_fingerprint(

        source_url=
            config.base_url,

        visible_text=
            soup.get_text(
                " ",
                strip=True
            ),

        discovered_urls=[
            candidate["source_url"]
            for candidate in candidates
        ],
    )


    candidates.sort(
        key=lambda item: (
            item["document_year"],
            item["source_url"]
        )
    )


    logger.info(
        "CNA: %s archivos candidatos encontrados.",
        len(candidates)
    )


    return candidates, fingerprint

In [0]:
# =============================================================================
# 08 - DESCUBRIMIENTO DE DOCUMENTOS CFN
# =============================================================================

def discover_cfn_files(
    target_years: Tuple[int, ...]
) -> Tuple[
    List[Dict[str, Any]],
    Dict[str, Any]
]:
    """
    Descubre documentos sectoriales relevantes
    publicados por la CFN.
    """

    config = SOURCES["CFN"]


    response = fetch_html(
        config
    )


    soup = BeautifulSoup(
        response.text,
        "lxml"
    )


    normalized_keywords = tuple(
        normalize_text(keyword)
        for keyword
        in config.keywords
    )


    candidates: List[
        Dict[str, Any]
    ] = []


    seen_urls = set()


    for anchor in soup.find_all(
        "a",
        href=True
    ):

        href = anchor.get(
            "href"
        )


        if not href:
            continue


        full_url = urljoin(
            config.base_url,
            href
        )


        parsed_path = unquote(
            urlparse(
                full_url
            ).path
        )


        extension = Path(
            parsed_path
        ).suffix.lower()


        if extension not in config.allowed_extensions:
            continue


        anchor_text = anchor.get_text(
            " ",
            strip=True
        )


        combined = normalize_text(
            f"{full_url} {anchor_text}"
        )


        # Relevancia temática
        if not any(
            keyword in combined
            for keyword
            in normalized_keywords
        ):
            continue


        document_year = infer_document_year(
            full_url,
            anchor_text
        )


        if document_year not in target_years:
            continue


        if full_url in seen_urls:
            continue


        seen_urls.add(
            full_url
        )


        candidates.append({

            "source":
        config.code,

    "source_page_url":
        config.base_url,

    "source_url":
        full_url,

    "resource_id":
        resource_id_from_url(
            full_url
        ),

    "anchor_text":
        anchor_text,

    "original_filename":
        Path(
            parsed_path
        ).name,

    "document_year":
        document_year,

    "extension":
        extension,
        })


    fingerprint = build_structural_fingerprint(

        source_url=
            config.base_url,

        visible_text=
            soup.get_text(
                " ",
                strip=True
            ),

        discovered_urls=[
            item["source_url"]
            for item in candidates
        ],
    )


    candidates.sort(
        key=lambda item: (
            item["document_year"],
            item["source_url"]
        )
    )


    logger.info(
        "CFN: %s archivos candidatos encontrados.",
        len(candidates)
    )


    return (
        candidates,
        fingerprint
    )

In [0]:
# =============================================================================
# 09 - MANIFIESTOS Y CONTROL HISTÓRICO
# =============================================================================

def get_latest_manifest(
    metadata_dir: Path,
    source_code: str
) -> Optional[Dict[str, Any]]:
    """
    Recupera el manifiesto más reciente de una fuente.
    """

    manifests = sorted(
        metadata_dir.glob(
            f"ingestion_manifest_*_{source_code}.json"
        ),
        key=lambda path:
            path.stat().st_mtime,
        reverse=True,
    )

    if not manifests:
        return None


    try:

        with open(
            manifests[0],
            "r",
            encoding="utf-8"
        ) as file_handle:

            payload = json.load(
                file_handle
            )


        payload["_manifest_path"] = str(
            manifests[0]
        )


        return payload


    except (
        OSError,
        json.JSONDecodeError
    ) as exc:

        logger.warning(
            "No se pudo leer el último manifiesto de %s: %s",
            source_code,
            exc,
        )

        return None


def build_previous_index(
    manifest: Optional[Dict[str, Any]]
) -> Dict[str, Dict[str, Any]]:
    """
    Indexa por URL los registros de la última
    ejecución compatible con el layout actual.
    """

    if not manifest:
        return {}


    # No reutilizar registros generados con
    # estructuras antiguas de almacenamiento.
    if (
        manifest.get(
            "storage_layout_version"
        )
        !=
        STORAGE_LAYOUT_VERSION
    ):

        logger.info(
            "Manifiesto anterior incompatible con "
            "storage layout %s. Se realizará "
            "adquisición completa.",
            STORAGE_LAYOUT_VERSION
        )

        return {}


    return {

        record["source_url"]:
            record

        for record
        in manifest.get(
            "files",
            []
        )

        if (
            record.get("source_url")
            and
            record.get("status")
            in {
                "downloaded",
                "updated_version",
                "unchanged",
            }
        )
    }


def compare_fingerprints(
    previous: Optional[Dict[str, Any]],
    current: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Compara la firma actual de la fuente
    con la firma registrada en la ejecución anterior.

    El cambio en el conjunto de URLs se considera
    el indicador principal de modificación relevante
    para la adquisición.
    """

    previous_fp = (
        previous or {}
    ).get(
        "structural_fingerprint"
    )


    if not previous_fp:

        return {

            "comparable":
                False,

            "changed":
                None,

            "url_set_changed":
                None,

            "visible_text_changed":
                None,
        }


    url_set_changed = (

        previous_fp.get(
            "url_set_sha256"
        )

        !=

        current.get(
            "url_set_sha256"
        )
    )


    visible_text_changed = (

        previous_fp.get(
            "visible_text_sha256"
        )

        !=

        current.get(
            "visible_text_sha256"
        )
    )


    return {

        "comparable":
            True,

        # El conjunto de documentos es el criterio
        # principal para la ingesta.
        "changed":
            url_set_changed,

        "url_set_changed":
            url_set_changed,

        # Se conserva como señal informativa porque
        # puede cambiar por contenido no relacionado.
        "visible_text_changed":
            visible_text_changed,
    }


def software_versions() -> Dict[str, str]:
    """
    Obtiene versiones relevantes para reproducibilidad.
    """

    def version(
        package_name: str
    ) -> str:

        try:

            return importlib_metadata.version(
                package_name
            )

        except importlib_metadata.PackageNotFoundError:

            return "unknown"


    return {

        "python":
            sys.version.split()[0],

        "requests":
            version("requests"),

        "beautifulsoup4":
            version("beautifulsoup4"),

        "lxml":
            version("lxml"),

        "tenacity":
            version("tenacity"),
    }


def write_manifest(
    *,
    source_code: str,
    batch_id: str,
    started_at_utc: str,
    finished_at_utc: str,
    fingerprint: Dict[str, Any],
    fingerprint_comparison: Dict[str, Any],
    records: List[Dict[str, Any]],
    metadata_dir: Path,
) -> Path:
    """
    Genera el manifiesto auditable de una ejecución Bronze.
    """

    counts = {

        "downloaded":
            sum(
                record.get("status")
                == "downloaded"
                for record in records
            ),

        "updated_version":
            sum(
                record.get("status")
                == "updated_version"
                for record in records
            ),

        "unchanged":
            sum(
                record.get("status")
                == "unchanged"
                for record in records
            ),

        "failed":
            sum(
                record.get("status")
                == "failed"
                for record in records
            ),
    }


    payload = {

        "manifest_schema_version":
            MANIFEST_SCHEMA_VERSION,

        "storage_layout_version":
            STORAGE_LAYOUT_VERSION,

        "notebook_version":
            NOTEBOOK_VERSION,

        "source":
            source_code,

        "source_page_url":
            SOURCES[
                source_code
            ].base_url,

        "batch_id":
            batch_id,

        "started_at_utc":
            started_at_utc,

        "finished_at_utc":
            finished_at_utc,

        "target_years":
            list(
                TARGET_YEARS
            ),

        "total_files":
            len(records),

        "status_counts":
            counts,

        "bytes_downloaded":
            sum(
                int(
                    record.get(
                        "size_bytes"
                    )
                    or 0
                )

                for record in records

                if record.get("status")
                in {
                    "downloaded",
                    "updated_version",
                }
            ),

        "files":
            sorted(
                records,
                key=lambda record: (
                    record.get(
                        "document_year"
                    )
                    or 0,
                    record.get(
                        "source_url"
                    )
                    or ""
                )
            ),

        "structural_fingerprint":
            fingerprint,

        "structural_fingerprint_comparison":
            fingerprint_comparison,

        "software_versions":
            software_versions(),
    }


    manifest_path = (
        metadata_dir
        /
        f"ingestion_manifest_"
        f"{batch_id}_"
        f"{source_code}.json"
    )


    atomic_json_dump(
        payload,
        manifest_path
    )


    logger.info(
        "Manifiesto generado: %s",
        manifest_path
    )


    return manifest_path

In [0]:
# =============================================================================
# 10 - DESCARGA, INTEGRIDAD, IDEMPOTENCIA Y VERSIONADO
# =============================================================================

@retry(
    stop=stop_after_attempt(4),

    wait=wait_exponential(
        multiplier=1,
        min=2,
        max=30
    ),

    retry=retry_if_exception_type(
        (
            requests.RequestException,
            OSError,
            ValueError,
        )
    ),

    before_sleep=before_sleep_log(
        logger,
        logging.WARNING
    ),

    reraise=True,
)
def download_candidate(
    candidate: Dict[str, Any],
    raw_dir: Path,
    previous_record: Optional[Dict[str, Any]],
    batch_id: str,
) -> Dict[str, Any]:
    """
    Descarga un documento de forma controlada.

    Propiedades:
    - identifica de forma única cada recurso por URL;
    - utiliza validadores HTTP cuando están disponibles;
    - calcula SHA-256 del contenido;
    - evita colisiones por nombre de archivo;
    - conserva versiones cuando cambia el contenido;
    - no sobrescribe versiones Bronze existentes.
    """

    checked_at_utc = utc_now_iso()


    time.sleep(
        random.uniform(
            *REQUEST_DELAY_SECONDS
        )
    )


    session = get_http_session()


    # -------------------------------------------------------------------------
    # Identificador determinista del recurso
    # -------------------------------------------------------------------------

    resource_id = (
        candidate.get(
            "resource_id"
        )
        or
        resource_id_from_url(
            candidate["source_url"]
        )
    )


    # -------------------------------------------------------------------------
    # Headers condicionales
    # -------------------------------------------------------------------------

    conditional_headers: Dict[str, str] = {}


    if previous_record:

        previous_etag = (
            previous_record.get(
                "etag"
            )
        )

        previous_last_modified = (
            previous_record.get(
                "last_modified"
            )
        )


        if previous_etag:

            conditional_headers[
                "If-None-Match"
            ] = previous_etag


        if previous_last_modified:

            conditional_headers[
                "If-Modified-Since"
            ] = previous_last_modified


    response = session.get(
        candidate["source_url"],
        headers=conditional_headers,
        stream=True,
        timeout=REQUEST_TIMEOUT,
    )


    # -------------------------------------------------------------------------
    # HTTP 304: recurso sin modificación
    # -------------------------------------------------------------------------

    if (
        response.status_code == 304
        and
        previous_record
    ):

        previous_stored_path = (
            previous_record.get(
                "stored_path"
            )
        )


        if previous_stored_path:

            previous_path = Path(
                previous_stored_path
            )


            if previous_path.exists():

                response.close()


                return {

                    **candidate,

                    "resource_id":
                        resource_id,

                    "batch_id":
                        batch_id,

                    "status":
                        "unchanged",

                    "stored_filename":
                        previous_record.get(
                            "stored_filename"
                        ),

                    "stored_path":
                        str(
                            previous_path
                        ),

                    "size_bytes":
                        previous_record.get(
                            "size_bytes"
                        ),

                    "sha256":
                        previous_record.get(
                            "sha256"
                        ),

                    "content_type":
                        previous_record.get(
                            "content_type"
                        ),

                    "etag":
                        previous_record.get(
                            "etag"
                        ),

                    "last_modified":
                        previous_record.get(
                            "last_modified"
                        ),

                    # Momento de la comprobación actual
                    "checked_at_utc":
                        checked_at_utc,

                    # Se conserva el timestamp de la
                    # descarga material real.
                    "downloaded_at_utc":
                        previous_record.get(
                            "downloaded_at_utc"
                        ),
                }


        # El servidor indica 304 pero la copia local
        # ya no está disponible. Se fuerza GET completo.
        response.close()


        response = session.get(
            candidate["source_url"],
            stream=True,
            timeout=REQUEST_TIMEOUT,
        )


    response.raise_for_status()


    # -------------------------------------------------------------------------
    # Directorio por año documental
    # -------------------------------------------------------------------------

    document_year = (
        candidate.get(
            "document_year"
        )
        or
        "unknown"
    )


    year_dir = (
        raw_dir
        /
        str(document_year)
    )


    year_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # -------------------------------------------------------------------------
    # Nombre original
    # -------------------------------------------------------------------------

    original_filename = (

        candidate.get(
            "original_filename"
        )

        or

        Path(
            urlparse(
                candidate["source_url"]
            ).path
        ).name
    )


    safe_name = safe_filename(
        original_filename
    )


    source_path = Path(
        safe_name
    )


    stem = source_path.stem

    suffix = source_path.suffix.lower()


    # -------------------------------------------------------------------------
    # Archivo temporal ÚNICO POR URL
    # -------------------------------------------------------------------------

    temp_path = (
        year_dir
        /
        (
            f".{stem}"
            f"__src_{resource_id}"
            f"__{batch_id}"
            f".part"
        )
    )


    try:

        with open(
            temp_path,
            "wb"
        ) as file_handle:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):

                if chunk:

                    file_handle.write(
                        chunk
                    )


        # ---------------------------------------------------------------------
        # Validar contenido no vacío
        # ---------------------------------------------------------------------

        if (
            not temp_path.exists()
            or
            temp_path.stat().st_size == 0
        ):

            raise ValueError(
                "El servidor devolvió un archivo vacío: "
                f"{candidate['source_url']}"
            )


        # ---------------------------------------------------------------------
        # Integridad
        # ---------------------------------------------------------------------

        sha256, size_bytes = (
            compute_sha256_and_size(
                temp_path
            )
        )


        # ---------------------------------------------------------------------
        # Comprobación Content-Length cuando es aplicable
        # ---------------------------------------------------------------------

        declared_length = (
            response.headers.get(
                "content-length"
            )
        )

        content_encoding = (
            response.headers.get(
                "content-encoding"
            )
        )


        if (
            declared_length
            and
            declared_length.isdigit()
            and
            not content_encoding
            and
            int(declared_length)
            != size_bytes
        ):

            raise ValueError(
                f"Tamaño descargado ({size_bytes}) "
                f"distinto de Content-Length "
                f"({declared_length}) para "
                f"{candidate['source_url']}"
            )


        # ---------------------------------------------------------------------
        # Nombre físico inmutable
        # ---------------------------------------------------------------------

        stored_filename = (
            f"{stem}"
            f"__src_{resource_id}"
            f"__sha256_{sha256[:12]}"
            f"{suffix}"
        )


        final_path = (
            year_dir
            /
            stored_filename
        )


        # ---------------------------------------------------------------------
        # El mismo contenido ya existe
        # ---------------------------------------------------------------------

        if final_path.exists():

            existing_sha256, existing_size = (
                compute_sha256_and_size(
                    final_path
                )
            )


            if existing_sha256 != sha256:

                raise RuntimeError(
                    "Se detectó inconsistencia crítica: "
                    "el nombre versionado corresponde al mismo SHA-256 "
                    "abreviado pero el contenido físico no coincide."
                )


            temp_path.unlink(
                missing_ok=True
            )


            return {

                **candidate,

                "resource_id":
                    resource_id,

                "batch_id":
                    batch_id,

                "status":
                    "unchanged",

                "stored_filename":
                    final_path.name,

                "stored_path":
                    str(
                        final_path
                    ),

                "size_bytes":
                    existing_size,

                "sha256":
                    existing_sha256,

                "content_type":
                    response.headers.get(
                        "content-type"
                    ),

                "etag":
                    response.headers.get(
                        "etag"
                    ),

                "last_modified":
                    response.headers.get(
                        "last-modified"
                    ),

                "checked_at_utc":
                    checked_at_utc,

                "downloaded_at_utc":
                    (
                        previous_record.get(
                            "downloaded_at_utc"
                        )
                        if previous_record
                        else checked_at_utc
                    ),
            }


        # ---------------------------------------------------------------------
        # Determinar si es archivo nuevo o nueva versión
        # ---------------------------------------------------------------------

        previous_sha256 = (
            previous_record.get(
                "sha256"
            )
            if previous_record
            else None
        )


        if (
            previous_sha256
            and
            previous_sha256 != sha256
        ):

            status = "updated_version"

        else:

            status = "downloaded"


        # ---------------------------------------------------------------------
        # Persistencia inmutable
        # ---------------------------------------------------------------------

        temp_path.replace(
            final_path
        )


        logger.info(
            "%s | %s | %s | %.2f KiB",
            candidate["source"],
            status,
            final_path.name,
            size_bytes / 1024,
        )


        return {

            **candidate,

            "resource_id":
                resource_id,

            "batch_id":
                batch_id,

            "status":
                status,

            "stored_filename":
                final_path.name,

            "stored_path":
                str(
                    final_path
                ),

            "size_bytes":
                size_bytes,

            "sha256":
                sha256,

            "content_type":
                response.headers.get(
                    "content-type"
                ),

            "etag":
                response.headers.get(
                    "etag"
                ),

            "last_modified":
                response.headers.get(
                    "last-modified"
                ),

            "checked_at_utc":
                checked_at_utc,

            # Hubo transferencia real del documento.
            "downloaded_at_utc":
                utc_now_iso(),
        }


    except Exception:

        temp_path.unlink(
            missing_ok=True
        )

        raise


    finally:

        response.close()

In [0]:
# =============================================================================
# 11 - COORDINACIÓN DE INGESTA POR FUENTE
# =============================================================================

DISCOVERY_FUNCTIONS = {

    "CNA":
        discover_cna_files,

    "CFN":
        discover_cfn_files,
}


def process_source(
    source_code: str,
    batch_id: str
) -> Dict[str, Any]:
    """
    Coordina descubrimiento, descarga,
    versionado y generación del manifiesto
    de una fuente.
    """

    config = SOURCES[
        source_code
    ]

    paths = BRONZE_PATHS[
        source_code
    ]


    started_at = utc_now_iso()


    previous_manifest = get_latest_manifest(
        paths["metadata"],
        source_code
    )


    previous_index = build_previous_index(
        previous_manifest
    )


    # -------------------------------------------------------------------------
    # Descubrimiento
    # -------------------------------------------------------------------------

    try:

        candidates, fingerprint = (
            DISCOVERY_FUNCTIONS[
                source_code
            ](
                TARGET_YEARS
            )
        )


    except Exception as exc:

        logger.exception(
            "%s: error durante el descubrimiento.",
            source_code
        )


        fingerprint = {

            "source_url":
                config.base_url,

            "captured_at_utc":
                utc_now_iso(),

            "error":
                str(exc),
        }


        records = [{

            "source":
                source_code,

            "source_page_url":
                config.base_url,

            "source_url":
                config.base_url,

            "document_year":
                None,

            "status":
                "failed",

            "error_message":
                str(exc)[:1000],

            "batch_id":
                batch_id,

            "downloaded_at_utc":
                utc_now_iso(),
        }]


        comparison = compare_fingerprints(
            previous_manifest,
            fingerprint
        )


        finished_at = utc_now_iso()


        manifest_path = write_manifest(

            source_code=
                source_code,

            batch_id=
                batch_id,

            started_at_utc=
                started_at,

            finished_at_utc=
                finished_at,

            fingerprint=
                fingerprint,

            fingerprint_comparison=
                comparison,

            records=
                records,

            metadata_dir=
                paths["metadata"],
        )


        return {

            "source":
                source_code,

            "manifest_path":
                str(
                    manifest_path
                ),

            "records":
                records,

            "total_candidates":
                0,

            "failed":
                1,
        }


    comparison = compare_fingerprints(
        previous_manifest,
        fingerprint
    )


    records: List[
        Dict[str, Any]
    ] = []


    # -------------------------------------------------------------------------
    # Descarga concurrente
    # -------------------------------------------------------------------------

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:


        futures = {

            executor.submit(

                download_candidate,

                candidate,

                paths["raw"],

                previous_index.get(
                    candidate[
                        "source_url"
                    ]
                ),

                batch_id,

            ):
                candidate

            for candidate
            in candidates
        }


        for future in as_completed(
            futures
        ):


            candidate = futures[
                future
            ]


            try:

                record = future.result()


                logger.info(
                    "%s | %s | %s",
                    source_code,
                    record[
                        "status"
                    ],
                    record.get(
                        "stored_filename"
                    ),
                )


            except Exception as exc:

                logger.error(
                    "%s | failed | %s | %s",
                    source_code,
                    candidate[
                        "source_url"
                    ],
                    exc,
                )


                record = {

                    **candidate,

                    "batch_id":
                        batch_id,

                    "status":
                        "failed",

                    "error_message":
                        str(exc)[:1000],

                    "downloaded_at_utc":
                        utc_now_iso(),
                }


            records.append(
                record
            )


    # -------------------------------------------------------------------------
    # Manifiesto
    # -------------------------------------------------------------------------

    finished_at = utc_now_iso()


    manifest_path = write_manifest(

        source_code=
            source_code,

        batch_id=
            batch_id,

        started_at_utc=
            started_at,

        finished_at_utc=
            finished_at,

        fingerprint=
            fingerprint,

        fingerprint_comparison=
            comparison,

        records=
            records,

        metadata_dir=
            paths["metadata"],
    )


    return {

        "source":
            source_code,

        "manifest_path":
            str(
                manifest_path
            ),

        "records":
            records,

        "total_candidates":
            len(
                candidates
            ),

        "downloaded":
            sum(
                record.get(
                    "status"
                )
                == "downloaded"

                for record
                in records
            ),

        "updated_version":
            sum(
                record.get(
                    "status"
                )
                == "updated_version"

                for record
                in records
            ),

        "unchanged":
            sum(
                record.get(
                    "status"
                )
                == "unchanged"

                for record
                in records
            ),

        "failed":
            sum(
                record.get(
                    "status"
                )
                == "failed"

                for record
                in records
            ),

        "fingerprint_changed":
            comparison.get(
                "changed"
            ),
    }

In [0]:
# =============================================================================
# 12 - EJECUCIÓN PRINCIPAL
# =============================================================================

def main() -> Dict[str, Any]:
    """
    Ejecuta secuencialmente la ingesta
    de CNA y CFN dentro de un mismo batch.
    """

    batch_id = generate_batch_id()

    pipeline_started_at = utc_now_iso()


    logger.info(
        "=== INICIO INGESTA BRONZE | batch_id=%s ===",
        batch_id
    )


    results: Dict[
        str,
        Any
    ] = {}


    # Las fuentes se procesan secuencialmente.
    # La concurrencia se limita a las descargas
    # dentro de cada fuente.
    for source_code in (
        "CNA",
        "CFN"
    ):

        results[
            source_code
        ] = process_source(
            source_code,
            batch_id
        )


    totals = {

        "sources":
            len(results),

        "candidates":
            sum(
                result.get(
                    "total_candidates",
                    0
                )
                for result
                in results.values()
            ),

        "downloaded":
            sum(
                result.get(
                    "downloaded",
                    0
                )
                for result
                in results.values()
            ),

        "updated_version":
            sum(
                result.get(
                    "updated_version",
                    0
                )
                for result
                in results.values()
            ),

        "unchanged":
            sum(
                result.get(
                    "unchanged",
                    0
                )
                for result
                in results.values()
            ),

        "failed":
            sum(
                result.get(
                    "failed",
                    0
                )
                for result
                in results.values()
            ),
    }


    pipeline_finished_at = utc_now_iso()


    logger.info(

        "=== FIN INGESTA BRONZE | "
        "batch_id=%s | "
        "candidatos=%s | "
        "nuevos=%s | "
        "actualizados=%s | "
        "sin_cambios=%s | "
        "fallos=%s ===",

        batch_id,

        totals[
            "candidates"
        ],

        totals[
            "downloaded"
        ],

        totals[
            "updated_version"
        ],

        totals[
            "unchanged"
        ],

        totals[
            "failed"
        ],
    )


    return {

        "batch_id":
            batch_id,

        "started_at_utc":
            pipeline_started_at,

        "finished_at_utc":
            pipeline_finished_at,

        "totals":
            totals,

        "sources":
            results,
    }


result = main()

In [0]:
# =============================================================================
# 13 - VALIDACIÓN POST-INGESTA
# =============================================================================

def validate_record(
    record: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Verifica existencia física y SHA-256
    del archivo almacenado.
    """

    status = record.get(
        "status"
    )


    if status == "failed":

        return {

            "source":
                record.get(
                    "source"
                ),

            "stored_path":
                record.get(
                    "stored_path"
                ),

            "valid":
                False,

            "reason":
                record.get(
                    "error_message",
                    "download_failed"
                ),
        }


    stored_path = record.get(
        "stored_path"
    )

    expected_sha256 = record.get(
        "sha256"
    )


    if not stored_path:

        return {

            "source":
                record.get(
                    "source"
                ),

            "stored_path":
                None,

            "valid":
                False,

            "reason":
                "missing_stored_path",
        }


    path = Path(
        stored_path
    )


    if not path.exists():

        return {

            "source":
                record.get(
                    "source"
                ),

            "stored_path":
                stored_path,

            "valid":
                False,

            "reason":
                "file_not_found",
        }


    actual_sha256, actual_size = (
        compute_sha256_and_size(
            path
        )
    )


    valid = bool(
        expected_sha256
        and
        actual_sha256
        == expected_sha256
    )


    return {

        "source":
            record.get(
                "source"
            ),

        "stored_path":
            stored_path,

        "valid":
            valid,

        "reason":
            (
                None
                if valid
                else "sha256_mismatch"
            ),

        "expected_sha256":
            expected_sha256,

        "actual_sha256":
            actual_sha256,

        "size_bytes":
            actual_size,
    }


# -----------------------------------------------------------------------------
# Validar todos los registros del batch
# -----------------------------------------------------------------------------

validation_rows = []


for source_result in (
    result["sources"].values()
):

    for record in source_result.get(
        "records",
        []
    ):

        validation_rows.append(
            validate_record(
                record
            )
        )


invalid_rows = [

    row
    for row
    in validation_rows

    if not row[
        "valid"
    ]
]


valid_rows = [

    row
    for row
    in validation_rows

    if row[
        "valid"
    ]
]


# -----------------------------------------------------------------------------
# Resumen por fuente
# -----------------------------------------------------------------------------

summary_rows = []


for (
    source_code,
    source_result
) in result[
    "sources"
].items():


    summary_rows.append({

        "source":
            source_code,

        "candidates":
            int(
                source_result.get(
                    "total_candidates",
                    0
                )
            ),

        "downloaded":
            int(
                source_result.get(
                    "downloaded",
                    0
                )
            ),

        "updated_version":
            int(
                source_result.get(
                    "updated_version",
                    0
                )
            ),

        "unchanged":
            int(
                source_result.get(
                    "unchanged",
                    0
                )
            ),

        "failed":
            int(
                source_result.get(
                    "failed",
                    0
                )
            ),

        "fingerprint_changed":
            str(
                source_result.get(
                    "fingerprint_changed"
                )
            ),

        "manifest_path":
            str(
                source_result.get(
                    "manifest_path"
                )
            ),
    })


display(
    spark.createDataFrame(
        summary_rows
    )
)


logger.info(
    "Validación de integridad | válidos=%s | inválidos=%s",
    len(valid_rows),
    len(invalid_rows),
)


# -----------------------------------------------------------------------------
# Fallar explícitamente si la capa Bronze queda inconsistente
# -----------------------------------------------------------------------------

if invalid_rows:

    logger.error(
        "Se detectaron %s registros inválidos.",
        len(invalid_rows)
    )


    # Mostrar información completa antes de detener
    # el pipeline para facilitar el diagnóstico.
    display(
        spark.createDataFrame(
            invalid_rows
        )
    )


    for invalid in invalid_rows:

        logger.error(
            "Archivo inválido: %s | razón=%s",
            invalid.get(
                "stored_path"
            ),
            invalid.get(
                "reason"
            ),
        )


    raise RuntimeError(
        f"La validación Bronze detectó "
        f"{len(invalid_rows)} archivo(s) "
        f"inválido(s)."
    )


logger.info(
    "VALIDACIÓN BRONZE SUPERADA | "
    "archivos=%s | "
    "válidos=%s | "
    "inválidos=0",
    len(validation_rows),
    len(valid_rows),
)